# Semana 2 — Retornos Financieros

**Econolab Financial Economics**

---

## Objetivo
Pasar de precios a **información económica real**. Los precios cuentan una historia; los retornos la explican.

## Concepto clave
> Un precio de $150 no dice nada. Un retorno del +5% en una semana dice mucho.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from data.generate_data import generate_stock_prices
from src.returns import simple_returns, log_returns, annualized_return, annualized_volatility, sharpe_ratio, rolling_volatility

prices = generate_stock_prices()
asset = prices['TECH']  # trabajamos con un activo primero

print('Datos cargados ✓  |  Período:', prices.index[0].date(), '→', prices.index[-1].date())

## 1. Retornos simples vs logarítmicos

Hay dos formas de medir cuánto ganó (o perdió) un activo:

| Tipo | Fórmula | Cuándo usar |
|------|---------|-------------|
| Simple | $r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$ | Comparaciones cortas, comunicación |
| Logarítmico | $r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$ | Econometría, series de tiempo largas |

In [ ]:
ret_simple = simple_returns(asset)
ret_log    = log_returns(asset)

fig, axes = plt.subplots(2, 2, figsize=(13, 7))

# Precios
axes[0,0].plot(asset.index, asset.values, color='#2563EB', linewidth=1.2)
axes[0,0].set_title('Precio')
axes[0,0].set_ylabel('USD')

# Retornos simples
axes[0,1].plot(ret_simple.index, ret_simple.values, color='#16A34A', linewidth=0.8, alpha=0.8)
axes[0,1].axhline(0, color='black', linewidth=0.6)
axes[0,1].set_title('Retornos simples')
axes[0,1].set_ylabel('%')

# Distribución retornos simples
axes[1,0].hist(ret_simple, bins=50, color='#16A34A', alpha=0.7, edgecolor='white')
xmin, xmax = axes[1,0].get_xlim()
x = np.linspace(xmin, xmax, 100)
mu, std = ret_simple.mean(), ret_simple.std()
axes[1,0].plot(x, stats.norm.pdf(x, mu, std) * len(ret_simple) * (xmax-xmin)/50,
               color='red', linewidth=2, label='Normal teórica')
axes[1,0].legend()
axes[1,0].set_title('Distribución (simples)')

# Distribución retornos log
axes[1,1].hist(ret_log, bins=50, color='#7C3AED', alpha=0.7, edgecolor='white')
mu2, std2 = ret_log.mean(), ret_log.std()
x2 = np.linspace(ret_log.min(), ret_log.max(), 100)
axes[1,1].plot(x2, stats.norm.pdf(x2, mu2, std2) * len(ret_log) * (ret_log.max()-ret_log.min())/50,
               color='red', linewidth=2, label='Normal teórica')
axes[1,1].legend()
axes[1,1].set_title('Distribución (logarítmicos)')

for ax in axes.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('TECH — Análisis de Retornos', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Estadísticas de retornos

In [ ]:
from src.returns import return_summary

# Resumen completo de todos los activos
summaries = [return_summary(prices[col], name=col) for col in prices.columns]
summary_df = pd.DataFrame(summaries).set_index('Activo')
summary_df

## 3. Volatilidad dinámica (clustering)

Uno de los **hechos estilizados** más importantes de los mercados financieros: los períodos de alta volatilidad se agrupan. Esto es el "volatility clustering".

In [ ]:
roll_vol = rolling_volatility(ret_log, window=21)  # ventana de 1 mes

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax1.plot(ret_log.index, ret_log.values, color='#2563EB', linewidth=0.7, alpha=0.8)
ax1.axhline(0, color='black', linewidth=0.5)
ax1.set_title('Retornos diarios', fontweight='bold')
ax1.set_ylabel('Retorno log')

ax2.plot(roll_vol.index, roll_vol.values, color='#DC2626', linewidth=1.5)
ax2.fill_between(roll_vol.index, roll_vol.values, alpha=0.2, color='#DC2626')
ax2.set_title('Volatilidad móvil 21 días (anualizada)', fontweight='bold')
ax2.set_ylabel('Volatilidad anual')

for ax in [ax1, ax2]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('TECH — Retornos y Volatilidad Dinámica', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 📝 Ejercicio

1. Calcula el Sharpe Ratio de todos los activos. ¿Cuál tiene la mejor relación riesgo-retorno?
2. ¿La distribución de retornos es exactamente normal? ¿Qué implicaciones tiene esto para medir riesgo?
3. Interpreta el Skewness y Kurtosis de ENERGY en términos económicos.

In [ ]:
# TODO: calcula el Sharpe Ratio de cada activo usando src.returns.sharpe_ratio
# y ordénalos de mejor a peor


**Tu interpretación:** *(2-3 oraciones)*